# Modelado Predictivo: Clasificación con K-Nearest Neighbors
**Proyecto:** Clasificador de Salud Fetal (Dataset Cardiotocography - UCI).

**Contexto:** Tomando como base el dataset limpio, libre de fugas de datos ("X_features.csv" y "y_target.csv"), este notebook implementa y evalúa el modelo **K-Nearest Neighbors** construido desde cero. KNN es un algoritmo de *aprendizaje perezoso (lazy learning)* que no construye una ecuación interna, sino que memoriza el dataset de entrenamiento y clasifica a los nuevos datos basándose en la etiqueta mayoritaria de sus K vecinos más cercanos utilizando la distancia Euclidiana.

### Objetivos del Modelado

1. **Construcción y Búsqueda Vecinal:** Instanciar el clasificador KNNClassifier programado a mano con ayuda del curso 'Machine_Learning_desde_Cero-11', que calculará la distancia geométrica exacta entre un nuevo registro cardiotocográfico y los  pacientes del conjunto de entrenamiento.

2. **Evaluación de Probabilidades:** Utilizar la proporción de votos de los vecinos más cercanos para generar una distribución de probabilidad continua 'predict_proba', indispensable para el cálculo del AUC.

3. **Análisis Clínico y Costo Computacional:** Comparar las métricas de rendimiento (F1-Score, Recall, Matriz de Confusión y ROC-AUC) frente a scikit-learn, analizando la viabilidad del modelo casero en la fase de predicción debido al alto costo computacional del cálculo de distancias por fuerza bruta.


### Ficha Metodológica del Modelo

* **Descripción breve del método:** Algoritmo no paramétrico que clasifica un paciente buscando a sus K vecinos más parecidos (menor distancia Euclidiana) en el conjunto de entrenamiento. El diagnóstico final se toma por votación mayoritaria.

* **Justificación de su uso:** Es el método más intuitivo en medicina basada en evidencia: *"Si el monitor de este paciente se comporta matemáticamente igual al de otros 5 pacientes pasados que tuvieron hipoxia, lo más probable es que este paciente también la tenga"*. Dado que depende de distancias, **la estandarización previa de los datos (Z-Score) es un paso obligatorio** para evitar que variables grandes (como los latidos base LB) dominen.

* **Métricas de evaluación utilizadas:** Exactitud Global (*Accuracy*), Matriz de Confusión, *Recall*, *Precisión*, *F1-Score* por clase, *Promedio Macro F1-Score* y el cálculo del *Área Bajo la Curva ROC (AUC)*.

* **Análisis de resultados:** Detallado al final del notebook, resaltando la sensibilidad obtenida en la clase patológica.

In [1]:
# Importación de librerías esenciales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# Modificación para importar los modelos fuera de la carpeta actual
import os
import sys

root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)

# Importación del modelo y métricas caseras
from src.Knn import KNNClassifier
from src.utils_and_metrics import train_test_split, standardize, accuracy, confusion_matrix, recall, f1_score, precision, roc

#### **1. Preparación de Datos: Partición y Estandarización Z-Score**
Para que la distancia Euclidiana funcione correctamente y no le dé un peso desproporcionado a variables con números grandes, debemos escalar las 21 características usando el método standardize. Para evitar fugas de información clínica, la media y desviación estándar se aprenden **solo** del conjunto de entrenamiento.

In [10]:
# Carga directa de matrices generadas en 02_EDA_Visualizacion
X_df = pd.read_csv('../data/processed/X_features.csv')
y_df = pd.read_csv('../data/processed/y_target.csv')

X = X_df.to_numpy()
y = y_df['NSP'].to_numpy()
 

# Partición de datos (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, seed=42)

# Estandarización Z-Score

X_train, mean_train, std_train = standardize(X_train)
X_test = (X_test - mean_train) / std_train

print(f"Dimensión de Matriz X Estandarizada: Entrenamiento={X_train.shape}, Prueba={X_test.shape}")
print(f"Entrenamiento: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Prueba:        X_test={X_test.shape}, y_test={y_test.shape}")
print(f"Clases en y:   {np.unique(y)}")

Dimensión de Matriz X Estandarizada: Entrenamiento=(1700, 21), Prueba=(426, 21)
Entrenamiento: X_train=(1700, 21), y_train=(1700,)
Prueba:        X_test=(426, 21), y_test=(426,)
Clases en y:   [1 2 3]


#### **2. "Entrenamiento" y Evaluación de KNN**


In [ ]:
# Instanciamos el modelo
knn_custom = KNNClassifier(n_neighbors=5)

# Entrenamiento
t0 = time.perf_counter()
knn_custom.fit(X_train, y_train)
tiempo_train_knn = time.perf_counter() - t0

# Predicción
t0 = time.perf_counter()
predicciones_prueba = knn_custom.predict(X_test)
tiempo_pred_knn = time.perf_counter() - t0

# Cálculo de métricas
n_classes = 3
matriz_test = confusion_matrix(y_test - 1, predicciones_prueba - 1, n_classes)
acc_test = accuracy(y_test, predicciones_prueba)
precisions_test = precision(matriz_test)
recalls_test = recall(matriz_test)
f1s_test = f1_score(matriz_test)

# Impresión de Evaluación Final
print("\n--- Resultados de Rendimiento en Prueba (Custom KNN) ---")
print(f"Exactitud Global (Accuracy):         {acc_test * 100:.2f}%")
print(f"Tiempo de entrenamiento (Memorizar): {tiempo_train_knn:.4f} s")
print(f"Tiempo de predicción (Cálculo):      {tiempo_pred_knn:.4f} s\n")

print("--- Métricas detalladas por Clase ---")
nombres_clases = ["1 (Normal)", "2 (Sospechoso)", "3 (Patológico)"]
for i, nombre in enumerate(nombres_clases):
    print(f"Clase {nombre:14s} -> Precision: {precisions_test[i]*100:.2f}% | Recall: {recalls_test[i]*100:.2f}% | F1-Score: {f1s_test[i]*100:.2f}%")

print(f"\nPromedio Macro F1-Score: {np.mean(f1s_test)*100:.2f}%")
print("\nMatriz de Confusión (Filas: Real [1,2,3] | Columnas: Predicho [1,2,3]):")
print(matriz_test)


--- Resultados de Rendimiento en Prueba (Custom KNN) ---
Exactitud Global (Accuracy):         89.67%
Tiempo de entrenamiento (Memorizar): 0.0003 s
Tiempo de predicción (Cálculo):      6.6227 s

--- Métricas detalladas por Clase ---
Clase 1 (Normal)     -> Precision: 91.83% | Recall: 97.31% | F1-Score: 94.49%
Clase 2 (Sospechoso) -> Precision: 76.00% | Recall: 58.46% | F1-Score: 66.09%
Clase 3 (Patológico) -> Precision: 85.71% | Recall: 69.23% | F1-Score: 76.60%

Promedio Macro F1-Score: 79.06%

Matriz de Confusión (Filas: Real [1,2,3] | Columnas: Predicho [1,2,3]):
[[326   8   1]
 [ 25  38   2]
 [  4   4  18]]


In [ ]:
# Gráfica de Curva ROC Multiclase (One-vs-Rest)
# Extraemos la matriz de probabilidades de votos para la función matemática del AUC
macro_auc_knn, aucs_por_clase_knn = roc(knn_custom, X_test, y_test)
print(f"Macro AUC Global (Modelo Propio): {macro_auc_knn:.4f}")

#### **3. Comparación con Modelo de Scikit-Learn (KNeighborsClassifier)**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier as SklearnKNN
from sklearn.metrics import roc_auc_score

# Ajuste y configuración equivalente
knn_sklearn = SklearnKNN(
    n_neighbors=5,
    n_jobs=-1
)

t0 = time.perf_counter()
knn_sklearn.fit(X_train, y_train)
tiempo_train_sk = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_sk = knn_sklearn.predict(X_test)
tiempo_pred_sk = time.perf_counter() - t0

y_proba_sk = knn_sklearn.predict_proba(X_test)

# Métricas Sklearn
conf_matrix_sk = confusion_matrix(y_test - 1, y_pred_sk - 1, n_classes=3)
f1s_sk = f1_score(conf_matrix_sk)

print("--- Rendimiento KNN (Scikit-Learn) ---")
print(f"-> Exactitud Global en Prueba (Test): {accuracy(y_test, y_pred_sk) * 100:.2f}%")
print(f"-> Tiempo Entrenamiento / Predicción: {tiempo_train_sk:.4f} s / {tiempo_pred_sk:.4f} s")
print(f"-> F1-Score:                    {f1s_sk}")
print(f"-> Recall:                      {recall(conf_matrix_sk)}")
print(f"-> Precision:                   {precision(conf_matrix_sk)}")
print(f"-> Promedio Macro F1-Score:     {np.mean(f1s_sk) * 100:.2f}%")
print("\nMatriz de Confusión (Filas: Real | Columnas: Predicho):")
print(conf_matrix_sk)

# ROC
macro_auc_sk, aucs_por_clase_sk = roc(knn_sklearn, X_test, y_test)
print(f"Macro AUC Global (Scikit-Learn): {macro_auc_sk:.4f}")

#### **4. Análisis y Conclusiones Finales (Modelo Propio vs. Scikit-Learn)**

Al comparar nuestro modelo **K-Nearest Neighbors** creado desde cero con la versión de `scikit-learn` bajo el mismo número de vecinos (`K=5`), observamos un rendimiento clínico moderado pero descubrimos una barrera tecnológica crítica.

* **El desafío computacional del algoritmo:** A diferencia de Random Forest, donde lo difícil es entrenar, KNN entrena al instante (0.00 s), pero la predicción de nuestro código hecho a mano demoró casi 6 segundos, contrastando con la rapidez inmediata (0.06 s) de `scikit-learn`. Esto ocurre porque nuestro modelo mide la distancia por *fuerza bruta* contra los 1,700 pacientes de la base, mientras que la librería comercial utiliza complejas estructuras de datos optimizadas (como *KD-Trees* o *Ball Trees*) que buscan vecinos rápidamente sin iterar por toda la tabla.
* **Calidad Clínica y ROC:** Ambas implementaciones entregan exactamente la misma matriz de probabilidades (`predict_proba`), lo que asegura que nuestro algoritmo matemático y el cálculo del Área Bajo la Curva (**Macro AUC ~0.89**) fueron correctos. 
* **Sensibilidad en Hipoxia:** A pesar de haber escalado las variables (Z-Score), este modelo basado en distancias fue el que peor se desempeñó aislando a los pacientes con riesgo patológico inminente, logrando apenas un Recall moderado en la Clase 3.
* **Conclusión Final:** KNN es útil para casos diagnósticos altamente estructurados, pero en bioseñales complejas donde importan más las interacciones ocultas entre las 21 métricas (como los residuos corregidos en *Gradient Boosting*), KNN se queda corto clínica y computacionalmente.